# 1: Imports


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import coo_matrix


# 2: Đọc dữ liệu

In [ ]:
USE_DRIVE = False  # Nếu dùng Google Drive thì đặt True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path("/content/drive/MyDrive/epg_project")  # sửa lại cho đúng thư mục của bạn
else:
    DATA_DIR = Path("/content")  # nếu upload file trực tiếp trong Colab

TRAIN_PATH = DATA_DIR / "train_logs_program.parquet"
TEST_EPG_PATH = DATA_DIR / "test_epg_candidates.parquet"
SUB_PATH = DATA_DIR / "submission.csv"

print("Train:", TRAIN_PATH)
print("Test EPG:", TEST_EPG_PATH)
print("Submission:", SUB_PATH)


Train: /content/train_logs_program.parquet
Test EPG: /content/test_epg_candidates.parquet
Submission: /content/submission.csv


# 3: Đọc train_logs_program.parquet và chuẩn bị tương tác


In [ ]:


logs = pd.read_parquet(TRAIN_PATH)

# Loại tv_show_id = 0 nếu có (chương trình không liên quan)
if "tv_show_id" in logs.columns:
    logs = logs[logs["tv_show_id"] != 0]

required_cols = {"user_id", "tv_show_id", "duration_view"}
missing = required_cols - set(logs.columns)
if missing:
    raise ValueError(f"Thiếu các cột trong train_logs_program.parquet: {missing}")

logs["user_id"] = logs["user_id"].astype(np.int64)
logs["tv_show_id"] = logs["tv_show_id"].astype(np.int64)
logs["duration_view"] = logs["duration_view"].astype(np.float32)

# Gom theo (user, tv_show)
interactions = (
    logs
    .groupby(["user_id", "tv_show_id"], as_index=False)["duration_view"]
    .sum()
    .rename(columns={"duration_view": "watch_seconds"})
)

# Trọng số implicit: log(1 + số giây xem)
interactions["weight"] = np.log1p(interactions["watch_seconds"])

print("Số (user, tv_show) sau gom:", len(interactions))
print(interactions.head())


Số (user, tv_show) sau gom: 627456
               user_id  tv_show_id  watch_seconds    weight
0 -9218065423650594400       20088         7794.0  8.961238
1 -9218065423650594400      200337          885.0  6.786717
2 -9218065423650594400      200352         8170.0  9.008347
3 -9218065423650594400      200432         4514.0  8.415160
4 -9218065423650594400      240081         6893.0  8.838407


# 4: Encode user_id, tv_show_id và tạo ma trận user-item (CSR)


In [ ]:

unique_users = interactions["user_id"].unique()
unique_items = interactions["tv_show_id"].unique()

print("Số user:", len(unique_users))
print("Số item (tv_show):", len(unique_items))

user2idx = {u: i for i, u in enumerate(unique_users)}
idx2user = np.array(unique_users)

item2idx = {i: j for j, i in enumerate(unique_items)}
idx2item = np.array(unique_items)

rows = interactions["user_id"].map(user2idx).values
cols = interactions["tv_show_id"].map(item2idx).values
data = interactions["weight"].astype(np.float64).values   # dùng float64 cho ổn định

n_users = len(unique_users)
n_items = len(unique_items)

from scipy.sparse import csr_matrix

user_item = csr_matrix((data, (rows, cols)), shape=(n_users, n_items), dtype=np.float64)

print("Kích thước ma trận user_item:", user_item.shape)


Số user: 4885
Số item (tv_show): 4160
Kích thước ma trận user_item: (4885, 4160)


# 5: Cài đặt WMF (ALS)


In [ ]:

from scipy.sparse import csr_matrix, csc_matrix

class WMFImplicitALS:
    def __init__(self, n_factors=64, reg=0.02, alpha=40.0, n_iters=10, verbose=True):
        """
        n_factors: số chiều latent
        reg      : hệ số regularization
        alpha    : hệ số khuếch đại tín hiệu implicit
        n_iters  : số vòng lặp ALS
        """
        self.n_factors = n_factors
        self.reg = reg
        self.alpha = alpha
        self.n_iters = n_iters
        self.verbose = verbose

    def fit(self, R):
        """
        R: ma trận user-item dạng CSR, giá trị là weight > 0 (ví dụ: log1p(duration))
        """
        if not isinstance(R, csr_matrix):
            R = R.tocsr()
        self.n_users, self.n_items = R.shape

        self.user_factors = np.random.normal(scale=0.01, size=(self.n_users, self.n_factors))
        self.item_factors = np.random.normal(scale=0.01, size=(self.n_items, self.n_factors))

        R_csr = R.tocsr()
        R_csc = R.tocsc()

        I_f = np.eye(self.n_factors, dtype=np.float64)

        for it in range(self.n_iters):
            if self.verbose:
                print(f"ALS iteration {it+1}/{self.n_iters}")

            # ----- update user factors -----
            Y = self.item_factors
            YtY = Y.T @ Y

            for u in range(self.n_users):
                start, end = R_csr.indptr[u], R_csr.indptr[u+1]
                if start == end:
                    # user không có tương tác -> giữ vector ban đầu
                    continue
                item_idx = R_csr.indices[start:end]
                r_u = R_csr.data[start:end]

                A = YtY + self.reg * I_f
                b = np.zeros(self.n_factors, dtype=np.float64)

                # C_u = I + alpha * diag(r_u)
                for i, r_ui in zip(item_idx, r_u):
                    y_i = Y[i]
                    c_ui = 1.0 + self.alpha * r_ui
                    A += (c_ui - 1.0) * np.outer(y_i, y_i)
                    b += c_ui * y_i

                self.user_factors[u] = np.linalg.solve(A, b)

            # ----- update item factors -----
            X = self.user_factors
            XtX = X.T @ X

            for i in range(self.n_items):
                start, end = R_csc.indptr[i], R_csc.indptr[i+1]
                if start == end:
                    continue
                user_idx = R_csc.indices[start:end]
                r_i = R_csc.data[start:end]

                A = XtX + self.reg * I_f
                b = np.zeros(self.n_factors, dtype=np.float64)

                for u, r_ui in zip(user_idx, r_i):
                    x_u = X[u]
                    c_ui = 1.0 + self.alpha * r_ui
                    A += (c_ui - 1.0) * np.outer(x_u, x_u)
                    b += c_ui * x_u

                self.item_factors[i] = np.linalg.solve(A, b)

        return self


# 6: Train WMF


In [ ]:

wmf_model = WMFImplicitALS(
    n_factors=64,
    reg=0.02,
    alpha=40.0,
    n_iters=16,     # có thể tăng lên 10-15 nếu vẫn nhanh
    verbose=True
)

wmf_model.fit(user_item)

print("Train xong WMF.")


ALS iteration 1/16
ALS iteration 2/16
ALS iteration 3/16
ALS iteration 4/16
ALS iteration 5/16
ALS iteration 6/16
ALS iteration 7/16
ALS iteration 8/16
ALS iteration 9/16
ALS iteration 10/16
ALS iteration 11/16
ALS iteration 12/16
ALS iteration 13/16
ALS iteration 14/16
ALS iteration 15/16
ALS iteration 16/16
Train xong WMF.


# 7: Đọc test_epg_candidates và tạo candidate + fallback


In [ ]:

test_epg = pd.read_parquet(TEST_EPG_PATH)

if "tv_show_id" not in test_epg.columns:
    raise ValueError("test_epg_candidates.parquet phải có cột 'tv_show_id'.")

test_epg = test_epg[test_epg["tv_show_id"] != 0]

candidate_tv_ids = test_epg["tv_show_id"].astype(np.int64).unique().tolist()
candidate_tv_ids_set = set(candidate_tv_ids)

print("Số tv_show candidate trong test:", len(candidate_tv_ids))

# Tính độ phổ biến tv_show từ train (tổng weight)
tv_popularity = (
    interactions
    .groupby("tv_show_id")["weight"]
    .sum()
    .sort_values(ascending=False)
)

popular_tv_all = tv_popularity.index.tolist()
popular_tv_in_candidate = [tv for tv in popular_tv_all if tv in candidate_tv_ids_set]

FALLBACK_POOL = 100
if len(popular_tv_in_candidate) < FALLBACK_POOL:
    extra = [tv for tv in popular_tv_all if tv not in popular_tv_in_candidate]
    popular_tv_in_candidate = (popular_tv_in_candidate + extra)[:FALLBACK_POOL]
else:
    popular_tv_in_candidate = popular_tv_in_candidate[:FALLBACK_POOL]

print("Số tv_show dùng fallback:", len(popular_tv_in_candidate))


Số tv_show candidate trong test: 6636
Số tv_show dùng fallback: 100


# 8: Hàm recommend cho 1 user_id


In [ ]:

def recommend_wmf_for_user(user_id, topn=5):
    rec_tv = []

    if user_id in user2idx:
        uidx = user2idx[user_id]
        user_vec = wmf_model.user_factors[uidx]  # (n_factors,)
        item_vecs = wmf_model.item_factors       # (n_items, n_factors)

        scores = item_vecs @ user_vec  # (n_items,)
        # sort index theo score giảm dần
        item_order = np.argsort(-scores)

        for item_idx in item_order:
            tv = idx2item[item_idx]
            if tv in candidate_tv_ids_set and tv not in rec_tv:
                rec_tv.append(tv)
                if len(rec_tv) >= topn:
                    break

    # Fallback nếu user mới hoặc chưa đủ topn
    if len(rec_tv) < topn:
        for tv in popular_tv_in_candidate:
            if tv not in rec_tv:
                rec_tv.append(tv)
            if len(rec_tv) >= topn:
                break

    # đảm bảo đúng độ dài
    if len(rec_tv) < topn:
        while len(rec_tv) < topn:
            rec_tv.append(popular_tv_in_candidate[0])

    return rec_tv[:topn]


# Test nhanh với 1 user trong submission (sẽ thực sự chạy ở cell sau)
print("Ví dụ recommend:", recommend_wmf_for_user(interactions['user_id'].iloc[0], topn=5))


Ví dụ recommend: [np.int64(90076806), np.int64(90082493), np.int64(90041483), np.int64(90060882), np.int64(90059866)]


# 9: Sinh dự đoán và ghi vào cột tv_show_id


In [ ]:

# Đọc submission
submission = pd.read_csv(SUB_PATH)

# Kiểm tra cột
if "user_id" not in submission.columns or "tv_show_id" not in submission.columns:
    raise ValueError("submission.csv phải có 2 cột: 'user_id' và 'tv_show_id'.")

user_ids_submit = submission["user_id"].values

pred_strings = []  # mỗi phần tử sẽ là chuỗi "id1 id2 id3 id4 id5"

for uid in user_ids_submit:
    # Lấy top-5 tv_show_id, đã xếp từ #1 (score cao nhất) → #5 (thấp nhất trong top 5)
    preds = recommend_wmf_for_user(uid, topn=5)

    # Chuyển sang chuỗi "id1 id2 id3 id4 id5"
    pred_str = " ".join(str(int(tv)) for tv in preds)
    pred_strings.append(pred_str)

# Gán lại vào cột tv_show_id, thay thế "0 0 0 0 0"
submission["tv_show_id"] = pred_strings

# Lưu file submission mới
out_path = DATA_DIR / "submission_wmf_als.csv"
submission.to_csv(out_path, index=False)

print("Đã lưu file submission:", out_path)
submission.head()

Đã lưu file submission: /content/submission_wmf_als.csv


,user_id,tv_show_id
0,8377619604347126107,90074592 90057523 90076689 90053101 90057391
1,8381667675275833309,90049506 90065801 90075115 1000572 90073400
2,8387147770138767246,90079536 12002262 10002677 90057584 90025700
3,8397181578236218580,90080799 90064150 90049024 90080929 90077158
4,8404698046253197367,120078057 90075602 90079190 90035554 90073444
